In [1]:
import pandas as pd
cols_to_use = [
    'TARGET_B', # target
    'NGIFTALL', 'CARDGIFT', # frequency
    'AVGGIFT', 'MINRAMNT', 'MAXRAMNT', 'TIMELAG', # monetary
    'LASTDATE', 'FISTDATE', 'RFA_2R', # recency,
    'AGE', 'INCOME' # demographics
]
df = pd.read_csv('../data/cup98LRN.txt', usecols=cols_to_use)
df.shape

(95412, 12)

In [2]:
df.head()

,AGE,INCOME,NGIFTALL,CARDGIFT,MINRAMNT,MAXRAMNT,LASTDATE,FISTDATE,TIMELAG,AVGGIFT,TARGET_B,RFA_2R
0,60.0,NaN,31,14,5.0,12.0,9512,8911,4.0,7.741935,0,L
1,46.0,6.0,3,1,10.0,25.0,9512,9310,18.0,15.666667,0,L
2,NaN,3.0,27,14,2.0,16.0,9512,9001,12.0,7.481481,0,L
3,70.0,1.0,16,7,2.0,11.0,9512,8702,9.0,6.812500,0,L
4,78.0,3.0,37,8,3.0,15.0,9601,7903,14.0,6.864865,0,L


In [3]:
df['TARGET_B'].value_counts()

TARGET_B
0    90569
1     4843
Name: count, dtype: int64

In [4]:
missing_pct = df.isnull().mean().sort_values(ascending=False)
missing_pct.head(20)

AGE         0.248030
INCOME      0.223096
TIMELAG     0.104526
NGIFTALL    0.000000
CARDGIFT    0.000000
MINRAMNT    0.000000
LASTDATE    0.000000
MAXRAMNT    0.000000
FISTDATE    0.000000
AVGGIFT     0.000000
TARGET_B    0.000000
RFA_2R      0.000000
dtype: float64

In [5]:
(missing_pct > 0.9).sum()

np.int64(0)

In [6]:
missing_pct[missing_pct > 0.9]

Series([], dtype: float64)

In [7]:
df = df.drop(columns=missing_pct[missing_pct > 0.9].index)
df.shape

(95412, 12)

In [8]:
df['LASTDATE'].head(10)

0    9512
1    9512
2    9512
3    9512
4    9601
5    9506
6    9504
7    9508
8    9507
9    9504
Name: LASTDATE, dtype: int64

In [9]:
df['LASTDATE'].dtype

dtype('int64')

In [10]:
# Extract year/month from LASTDATE and FISTDATE
df['LASTDATE_YY'] = df['LASTDATE'] // 100
df['LASTDATE_MM'] = df['LASTDATE'] % 100
df['FISTDATE_YY'] = df['FISTDATE'] // 100
df['FISTDATE_MM'] = df['FISTDATE'] % 100

# Reference point: June 1997 (the 97NK mailing date)
ref_year, ref_month = 97, 6

# Months since last gift, and donor tenure (months since first gift)
df['MONTHS_SINCE_LAST_GIFT'] = (ref_year - df['LASTDATE_YY']) * 12 + (ref_month - df['LASTDATE_MM'])
df['MONTHS_SINCE_FIRST_GIFT'] = (ref_year - df['FISTDATE_YY']) * 12 + (ref_month - df['FISTDATE_MM'])

In [11]:
df[['LASTDATE', 'MONTHS_SINCE_LAST_GIFT', 'FISTDATE', 'MONTHS_SINCE_FIRST_GIFT']].head(10)

,LASTDATE,MONTHS_SINCE_LAST_GIFT,FISTDATE,MONTHS_SINCE_FIRST_GIFT
0,9512,18,8911,91
1,9512,18,9310,44
2,9512,18,9001,89
3,9512,18,8702,124
4,9601,17,7903,219
5,9506,24,9401,41
6,9504,26,8701,125
7,9508,22,9401,41
8,9507,23,8801,113
9,9504,26,9309,45


In [12]:
df[cols_to_use].isnull().sum()

TARGET_B        0
NGIFTALL        0
CARDGIFT        0
AVGGIFT         0
MINRAMNT        0
MAXRAMNT        0
TIMELAG      9973
LASTDATE        0
FISTDATE        0
RFA_2R          0
AGE         23665
INCOME      21286
dtype: int64

In [13]:
df[df['TIMELAG'].isnull()]['NGIFTALL'].describe()

count    9973.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: NGIFTALL, dtype: float64

In [14]:
df['TIMELAG'] = df['TIMELAG'].fillna(0)
df = df.drop(columns=['LASTDATE', 'FISTDATE', 'LASTDATE_YY', 'LASTDATE_MM', 'FISTDATE_YY', 'FISTDATE_MM'])
df.isnull().sum()

AGE                        23665
INCOME                     21286
NGIFTALL                       0
CARDGIFT                       0
MINRAMNT                       0
MAXRAMNT                       0
TIMELAG                        0
AVGGIFT                        0
TARGET_B                       0
RFA_2R                         0
MONTHS_SINCE_LAST_GIFT         0
MONTHS_SINCE_FIRST_GIFT        0
dtype: int64

In [15]:
df['AGE_MISSING'] = df['AGE'].isnull().astype(int)
df['AGE'] = df['AGE'].fillna(df['AGE'].median())

df['INCOME_MISSING'] = df['INCOME'].isnull().astype(int)
df['INCOME'] = df['INCOME'].fillna(df['INCOME'].median())

In [16]:
# Confirm no missing values remain anywhere
df.isnull().sum()

AGE                        0
INCOME                     0
NGIFTALL                   0
CARDGIFT                   0
MINRAMNT                   0
MAXRAMNT                   0
TIMELAG                    0
AVGGIFT                    0
TARGET_B                   0
RFA_2R                     0
MONTHS_SINCE_LAST_GIFT     0
MONTHS_SINCE_FIRST_GIFT    0
AGE_MISSING                0
INCOME_MISSING             0
dtype: int64

In [17]:
df['RFA_2R'].value_counts() # same value for every row --> useless

RFA_2R
L    95412
Name: count, dtype: int64

In [18]:
df = df.drop(columns=['RFA_2R']) # drop column

In [19]:
df.to_csv('../data/cup98_cleaned.csv', index=False) #save

In [20]:
X = df.drop(columns=['TARGET_B'])
y = df['TARGET_B']

X.shape, y.shape

((95412, 12), (95412,))

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((76329, 12), (19083, 12))

In [22]:
y_train.mean(), y_test.mean()

(np.float64(0.05075397293296126), np.float64(0.0507781795315202))

In [23]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

In [24]:
y_pred = model.predict(X_test) # make actual yes/no predictions on the data it's never seen.
y_pred_proba = model.predict_proba(X_test)[:, 1] #model's estimated probability of response for each donor (a number between 0 and 1). for ROC-AUC

In [25]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.60      0.74     18114
           1       0.07      0.56      0.12       969

    accuracy                           0.60     19083
   macro avg       0.52      0.58      0.43     19083
weighted avg       0.92      0.60      0.71     19083



In [26]:
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC: {roc_auc:.4f}")

ROC-AUC: 0.6036


In [27]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[10849  7265]
 [  425   544]]


In [28]:
threshold = 0.7  # raising threshold above the default 0.5
y_pred_adjusted = (y_pred_proba >= threshold).astype(int)

print(classification_report(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

           0       0.95      0.99      0.97     18114
           1       0.12      0.02      0.04       969

    accuracy                           0.94     19083
   macro avg       0.53      0.51      0.50     19083
weighted avg       0.91      0.94      0.92     19083



In [29]:
threshold = 0.6  # raising threshold above the default 0.5
y_pred_adjusted = (y_pred_proba >= threshold).astype(int)

print(classification_report(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

           0       0.95      0.92      0.94     18114
           1       0.10      0.15      0.12       969

    accuracy                           0.88     19083
   macro avg       0.52      0.54      0.53     19083
weighted avg       0.91      0.88      0.90     19083



In [30]:
threshold = 0.8  # raising threshold above the default 0.5
y_pred_adjusted = (y_pred_proba >= threshold).astype(int)

print(classification_report(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

           0       0.95      1.00      0.97     18114
           1       0.20      0.00      0.00       969

    accuracy                           0.95     19083
   macro avg       0.57      0.50      0.49     19083
weighted avg       0.91      0.95      0.92     19083



In [31]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, rf_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, rf_pred_proba):.4f}")

              precision    recall  f1-score   support

           0       0.95      0.99      0.97     18114
           1       0.03      0.01      0.01       969

    accuracy                           0.94     19083
   macro avg       0.49      0.50      0.49     19083
weighted avg       0.90      0.94      0.92     19083

ROC-AUC: 0.5670


In [32]:
#test random forest feature importance
import pandas as pd

importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False)

AVGGIFT                    0.148642
AGE                        0.142958
MONTHS_SINCE_FIRST_GIFT    0.132385
TIMELAG                    0.105609
MONTHS_SINCE_LAST_GIFT     0.089772
NGIFTALL                   0.085215
CARDGIFT                   0.075729
MAXRAMNT                   0.073338
INCOME                     0.066814
MINRAMNT                   0.047830
INCOME_MISSING             0.016114
AGE_MISSING                0.015594
dtype: float64

In [33]:
from xgboost import XGBClassifier

# scale_pos_weight approximates class_weight='balanced' for XGBoost specifically
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
xgb_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, xgb_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, xgb_pred_proba):.4f}")

              precision    recall  f1-score   support

           0       0.96      0.75      0.84     18114
           1       0.07      0.36      0.12       969

    accuracy                           0.73     19083
   macro avg       0.51      0.56      0.48     19083
weighted avg       0.91      0.73      0.80     19083

ROC-AUC: 0.5594


In [34]:
import pickle

with open('../api/model.pkl', 'wb') as f:
    pickle.dump(model, f) #use original logistic regression model

print("Model saved.")

Model saved.
